# Allen Base Panels
Generates inspection grid + paper panels (a), (b), (c) from `main_results/allen/modelsmc/sonnet`.

In [ ]:
import ast
import os
import sys
import types

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

# DSPy needs a cache dir before any modelsmc imports
os.environ["DSPY_CACHEDIR"] = "/tmp/dspy_cache_allen_panels"

# Inject dataclasses into sys.modules without triggering method/__init__.py
dataclasses_file = "../../../modelsmc/method/modules/dataclasses.py"
namespace = {}
with open(dataclasses_file) as f:
    exec(f.read(), namespace)
mock_module = types.ModuleType("modelsmc.method.modules.dataclasses")
mock_module.Particle = namespace["Particle"]
mock_module.ParticleHistory = namespace.get("ParticleHistory")
sys.modules["modelsmc.method.modules.dataclasses"] = mock_module

from modelsmc.tasks import AllenLevel0
from modelsmc.utils.plot_utils import use_style
from modelsmc.utils.plotting.plot_trajectory import plot_trajectories_raw
from modelsmc.utils.utils import convert_model_string

results_dir = "../../../main_results/allen/modelsmc/sonnet/"
fig_save_dir = os.path.join(os.getcwd(), "..", "panels")
os.makedirs(fig_save_dir, exist_ok=True)
print("fig_save_dir:", os.path.abspath(fig_save_dir))

---
## Load summary & build seed → run_id mapping

In [ ]:
summary_df = pd.read_csv(os.path.join(results_dir, "summary.csv"))

config_dict = ast.literal_eval(summary_df.iloc[0]["config"])
metric_to_optimize = config_dict["task"]["metric_to_optimize"]  # 'neg_log_marginal_NLE'
optimization_direction = config_dict["task"]["optimization_direction"]  # 'min'
print(f"metric: {metric_to_optimize}, direction: {optimization_direction}")

# Map seed -> run_id (each run_N folder corresponds to seed N)
seed_to_runid = {}
for run_id, grp in summary_df.groupby("run_id"):
    seed = ast.literal_eval(grp.iloc[0]["config"])["seed"]
    seed_to_runid[seed] = run_id
print("seed -> run_id:")
for s in sorted(seed_to_runid):
    print(f"  seed {s}: {seed_to_runid[s][:8]}...")

---
## Load task & real observations

In [ ]:
from omegaconf import OmegaConf

task_config = config_dict["task"].copy()
task_config["dir_data"] = "../../../modelsmc/tasks/allen/data/"
task_config["num_obs_train"] = 0
task_config["num_obs_valid"] = 10
task_config["plotting"] = False
task_config = OmegaConf.create(task_config)

allen_task = AllenLevel0(task_config)
_, valid_data = allen_task.get_data()
context_validation = allen_task.get_context(mode="validation")
simulation_wrapper = allen_task.simulation_wrapper

# 10 real observations (indices 0..9)
obs_real = list(valid_data)  # list of tensors, one per observation
print(f"Loaded {len(obs_real)} real observations, shape: {obs_real[0].shape}")

---
## Simulate: baseline + all 10 discovered runs

In [ ]:
n_runs = 10

_cache_baseline = os.path.join(
    os.path.dirname(os.path.abspath(".")), "allen", "obs_baseline_cache.pt"
)
_cache_discovered = os.path.join(
    os.path.dirname(os.path.abspath(".")), "allen", "obs_discovered_cache.pt"
)
# Store cache next to this notebook
_cache_baseline = os.path.join(os.getcwd(), "obs_baseline_cache.pt")
_cache_discovered = os.path.join(os.getcwd(), "obs_discovered_cache.pt")

if os.path.exists(_cache_baseline) and os.path.exists(_cache_discovered):
    print("Loading simulations from cache...")
    obs_baseline = torch.load(_cache_baseline, weights_only=False)
    obs_discovered = torch.load(_cache_discovered, weights_only=False)
    print(f"Baselines:   {obs_baseline.shape}")
    print(f"Discovered:  {obs_discovered.shape}")
else:

    def _load_and_simulate(run_dir):
        with open(os.path.join(run_dir, "simulator.py")) as f:
            sim_src = f.read()
        sim, _ = convert_model_string(sim_src)
        params = torch.load(
            os.path.join(run_dir, "parameter_estimates.pt"), weights_only=False
        )
        return simulation_wrapper(sim, params=params, context=context_validation)

    obs_baseline = []
    obs_discovered = []
    for seed in range(n_runs):
        print(f"Simulating run_{seed}...")
        obs_baseline.append(
            _load_and_simulate(os.path.join(results_dir, f"run_{seed}", "baseline"))
        )
        obs_discovered.append(
            _load_and_simulate(
                os.path.join(results_dir, f"run_{seed}", "best_particle")
            )
        )

    obs_baseline = torch.stack(obs_baseline)  # (n_runs, n_obs, T)
    obs_discovered = torch.stack(obs_discovered)  # (n_runs, n_obs, T)
    print(f"Baselines:   {obs_baseline.shape}")
    print(f"Discovered:  {obs_discovered.shape}")

    torch.save(obs_baseline, _cache_baseline)
    torch.save(obs_discovered, _cache_discovered)
    print(f"Saved cache to {_cache_baseline} and {_cache_discovered}")

---
## Inspection grid: real / baseline / run_0..9  ×  10 observations
Use this to pick `selected_run` in the next cell.

In [ ]:
n_obs = 10


def _strip(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


# One small figure per seed: 3 rows (real / baseline / discovered) × 10 obs
for seed in range(n_runs):
    fig, axes = plt.subplots(3, n_obs, figsize=(15, 5))

    for j in range(n_obs):
        axes[0, j].plot(obs_real[j], color="black", linewidth=0.5)
        axes[1, j].plot(obs_baseline[seed, j], color="darkred", linewidth=0.5)
        axes[2, j].plot(obs_discovered[seed, j], color="darkblue", linewidth=0.5)
        for row in range(3):
            axes[row, j].set_ylim(-110, 60)
            _strip(axes[row, j])

    for j in range(n_obs):
        axes[0, j].set_title(f"obs {j}", fontsize=6)

    plt.suptitle(f"Seed {seed}", fontsize=8, y=1.01)
    plt.tight_layout()
    plt.show()
    plt.close()

---
## Manual selection
Set `selected_run` and `plot_obs_ids` after inspecting the grid above.

In [ ]:
# selected_run = 1      # ← change after inspecting grid
# plot_obs_ids = [1, 5, 7]  # ← observation indices for panel (c)

# selected_run = 5      # ← change after inspecting grid
# plot_obs_ids = [0, 3, 8]  # ← observation indices for panel (c)

selected_run = 6  # ← change after inspecting grid
plot_obs_ids = [3, 7, 8]  # ← observation indices for panel (c)

---
## Panel (a): Convergence

In [ ]:
with open("panel_sizes_cm.yaml") as f:
    panel_sizes = yaml.safe_load(f)
scale = 90 / 72
fig_w = panel_sizes["panel_convergence"]["width_cm"] / 2.54 * scale
fig_h = panel_sizes["panel_convergence"]["height_cm"] / 2.54 * scale

max_time_step = 10

# Running-minimum best metric per run across iterations → mean ± 95 % CI band
run_curves = []
for run_id in seed_to_runid.values():
    grp = summary_df[summary_df["run_id"] == run_id][["iteration", metric_to_optimize]]
    best_so_far = np.inf
    curve = []
    for itr in range(max_time_step + 1):
        itr_vals = grp[grp["iteration"] == itr][metric_to_optimize]
        if len(itr_vals):
            best_so_far = min(best_so_far, itr_vals.min())
        curve.append(best_so_far)
    run_curves.append(curve)

run_curves = np.array(run_curves)  # (n_runs, max_time_step+1)
# Convert neg_log_marginal_NLE from total to per-observation average
n_obs_valid = task_config["num_obs_valid"]

mean = np.mean(run_curves, axis=0)
top = np.percentile(run_curves, 97.5, axis=0)
bot = np.percentile(run_curves, 2.5, axis=0)
iters = np.arange(max_time_step + 1)

with use_style("pyloric"):
    fig, ax = plt.subplots(1, 1, figsize=(fig_w, fig_h))

    plot_trajectories_raw(
        summary_csv_path=os.path.join(results_dir, "summary.csv"),
        pool_csv_path=os.path.join(results_dir, "pool_composition.csv"),
        run_id=seed_to_runid[selected_run],
        metric_name=metric_to_optimize,
        cmap_name=None,
        single_color="mediumblue",
        lw_full=1.5,
        s_new=3,
        alpha=0.2,
        fig=fig,
        ax=ax,
        max_time_step=max_time_step,
        show_failed_particles=False,
        show_copied_lines=False,
        deduplicate_markers=True,
    )

    ax.plot(iters, mean, color="darkblue", linewidth=1.5)
    ax.fill_between(iters, bot, top, color="darkblue", alpha=0.5, linewidth=0)

    ax.set_ylim(225, 280)
    ax.set_ylabel(r"$-\log p(\mathbf{x}_0 \mid m)$")
    # ax.set_xlabel("Iteration")

plt.tight_layout()
save_path = os.path.join(fig_save_dir, "convergence_allen.svg")
fig.savefig(save_path, bbox_inches="tight")
print(f"Saved {save_path}")
plt.show()
plt.close()

---
## Panel (b): Model channels per iteration
Manually annotate which ion channels are present at each iteration of `selected_run`.
Explicitly defined iterations are shown in black; inherited (unchanged) in light gray.

Explanation for this specific run and the different channels per iteration:
```markdown
# Iteration 1 (particle p8) — M-type K+ current (I_KM), voltage-independent tau
**New channel: M-type K+ (I_KM) via gating variable `p`**
- `gbar_M = params[:, 6]` (X1 slot, range [1e-4, 10] mS/cm²) — M-current max conductance
- Steady-state gate: `p_inf(V) = 1 / (1 + exp(-(V - (Vt + param_i)) / 9.0))`
  - Half-activation anchored to threshold: `v_half = Vt + param_i`
  - Slope factor 9 mV (physiologically standard for KCNQ/Kv7)
- Time constant: `tau_p = param_j` — constant (voltage-independent), range [1e-4, 3000] ms
  - Pre-computed once outside the loop (efficiency)
- Gate update: exponential Euler with fixed `tau_p_val`
- X2 slot (`params[:, 7]`) and `E_K` unchanged (`-107.0 mV`)
- `param_i`, `param_j` extracted **without negation** (correction vs. base template which negates them)
- M-current drives: `p * gbar_M` added to `tau_V_inv`; `p * gbar_M * E_K` added to `V_inf`

## Iteration 2 (particle p16) — M-type K+ current, corrected E_K
**Same M-type K+ channel as iter 1, plus one biophysical correction:**
- Channel structure identical: `p_inf(V)` sigmoid with `v_half = Vt + param_i`, slope 9 mV
- Time constant: `tau_p = param_j` — constant (voltage-independent), **computed inline per step** (not pre-cached)
- **Key change: `E_K` corrected from `-107.0` → `-90.0 mV`** (standard cortical HH value)
  - Comment in code: `-107.0 caused excessively hyperpolarized resting potential and distorted mean voltage, resting potential, skewness, and kurtosis statistics`
  - Applied to both delayed-rectifier K+ and M-current (both use `E_K`)
- Everything else structurally identical to p8
```

In [ ]:
with open("panel_sizes_cm.yaml") as f:
    panel_sizes = yaml.safe_load(f)
scale = 90 / 72
fig_w = panel_sizes["panel_model_parts"]["width_cm"] / 2.54 * scale
fig_h = panel_sizes["panel_model_parts"]["height_cm"] / 2.54 * scale

model_parts_values = ["leak", "na", "k", "m", "m_ek"]
model_parts_labels = ["Leak", r"Na$^{+}$", r"K$^{+}$", "M-K", r"M-K$^{E_K}$"]

# Fill in which channels are present at each explicit iteration;
# unspecified iterations inherit the last defined list.
# parts_present_explicit = {
#     0: ['leak', 'na', 'k'],
#     1: ['leak', 'na', 'k', 'm'],
#     2: ['leak', 'na', 'k', 'm_tau'],
#     3: ['leak', 'na', 'k', 'm_tau_tight'],
# }
parts_present_explicit = {
    0: ["leak", "na", "k"],
    1: ["leak", "na", "k", "m"],
    2: ["leak", "na", "k", "m_ek"],
}

parts_present = {}
parts_inherited = {}
last = []
for t in range(max_time_step + 1):
    if t in parts_present_explicit:
        parts_present[t] = parts_present_explicit[t]
        parts_inherited[t] = False
        last = parts_present_explicit[t]
    else:
        parts_present[t] = last.copy()
        parts_inherited[t] = True

with use_style("pyloric"):
    fig, ax = plt.subplots(1, 1, figsize=(fig_w, fig_h))

    for t, parts in parts_present.items():
        color = "lightgray" if parts_inherited[t] else "black"
        for part in parts:
            if part in model_parts_values:
                ax.plot(
                    t,
                    model_parts_values.index(part),
                    marker="x",
                    color=color,
                    markersize=5,
                    linestyle="none",
                )

    ax.set_yticks(range(len(model_parts_labels)))
    ax.set_yticklabels(model_parts_labels)
    ax.set_xlabel("Iteration")
    ax.set_xlim(-0.5, max_time_step + 0.5)

plt.tight_layout()
save_path = os.path.join(fig_save_dir, "model_parts_allen.svg")
fig.savefig(save_path, bbox_inches="tight")
print(f"Saved {save_path}")
plt.show()
plt.close()

---
## Panel (c): Posterior predictives
Three separate sub-figures: real observations, discovered predictives, baseline predictives.

In [ ]:
with open("panel_sizes_cm.yaml") as f:
    panel_sizes = yaml.safe_load(f)
scale = 90 / 72
fig_w = panel_sizes["panel_data"]["width_cm"] / 2.54 * scale
fig_h = panel_sizes["panel_data"]["height_cm"] / 2.54 * scale
n_plot = len(plot_obs_ids)


def _make_trace_fig(traces, color, suptitle, suptitle_x=0.825):
    """Plot `n_plot` voltage traces + blank spacer panel with suptitle."""
    with use_style("pyloric"):
        fig, axes = plt.subplots(1, n_plot + 1, figsize=(fig_w, fig_h))
        for idx, obs_id in enumerate(plot_obs_ids):
            axes[idx].plot(traces[obs_id], color=color, linewidth=0.5)
            axes[idx].set_ylim(-110, 60)
            axes[idx].set_xticks([])
            axes[idx].set_yticks([])
            for sp in axes[idx].spines.values():
                sp.set_visible(False)
        axes[-1].axis("off")
        plt.tight_layout()
        fig.suptitle(suptitle, ha="center", x=suptitle_x, y=0.7)
    return fig


# Real observations
fig_real = _make_trace_fig(obs_real, "black", "Real\nobservations")
path = os.path.join(fig_save_dir, "real_observations_allen.svg")
fig_real.savefig(path, bbox_inches="tight")
print(f"Saved {path}")
plt.show()
plt.close()

# Discovered predictives (selected run)
fig_disc = _make_trace_fig(
    obs_discovered[selected_run], "darkblue", "Discovered\npredictives"
)
path = os.path.join(fig_save_dir, "discovered_predictives_allen.svg")
fig_disc.savefig(path, bbox_inches="tight")
print(f"Saved {path}")
plt.show()
plt.close()

# Baseline predictives
fig_base = _make_trace_fig(
    obs_baseline[selected_run], "darkred", "Baseline\npredictives"
)
path = os.path.join(fig_save_dir, "baseline_predictives_allen.svg")
fig_base.savefig(path, bbox_inches="tight")
print(f"Saved {path}")
plt.show()
plt.close()